In [1]:
!git clone -b fine_tuning https://github.com/alapedriza1/mistral-7b-enterprise-function-calling.git

Cloning into 'mistral-7b-enterprise-function-calling'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 228 (delta 27), reused 32 (delta 17), pack-reused 167 (from 1)
Receiving objects: 100% (228/228), 811.92 KiB | 3.64 MiB/s, done.
Resolving deltas: 100% (134/134), done.


# 03 - Fine-Tuning

Fine-tunes Mistral 7B Instruct v0.3 on our synthetic function-calling dataset using QLoRA. Trains the model to reliably select the correct tool and produce valid JSON arguments for 16 enterprise tool schemas.

**Goal**: Improve structured function-calling accuracy over the baseline established in Notebook 02.

In [2]:
%pip install -q transformers accelerate bitsandbytes peft trl datasets tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 44.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))

# Verify login
user_info = whoami()
print(f"Logged in as: {user_info['name']}")

Logged in as: alapedriza


In [4]:
import sys
import os
import pandas as pd

# Add project root to path
PROJECT_ROOT = "/kaggle/working/mistral-7b-enterprise-function-calling"
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_jsonl, spot_check
from src.schemas import MAX_SEQ_LENGTH
from src.training import (
    load_model_for_training,
    apply_lora,
    run_training,
    check_truncation,
    merge_system_into_user,
    DEFAULT_LORA_CONFIG,
    DEFAULT_TRAINING_ARGS,
)

In [5]:
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/qlora-run-1"

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data = load_jsonl(f"{DATA_DIR}/val.jsonl")

print(f"Train: {len(train_data)} examples")
print(f"Val:   {len(val_data)} examples")

Train: 1228 examples
Val:   154 examples


In [6]:
model, tokenizer = load_model_for_training()
model = apply_lora(model)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [7]:
# Check if any examples will be truncated at current MAX_SEQ_LENGTH.
# Review the output — cancel execution if truncation is unacceptable.
train_stats = check_truncation(train_data, tokenizer, MAX_SEQ_LENGTH, label="train")
val_stats = check_truncation(val_data, tokenizer, MAX_SEQ_LENGTH, label="val")
stats = pd.concat([train_stats, val_stats], ignore_index=True)
stats.head()

,split,total,truncated,truncated_pct,mean_tokens,p95_tokens,max_tokens,max_over
0,train,1228,0,0.0,1063,1458,1715,0
1,val,154,0,0.0,1050,1444,1620,0


In [8]:
trainer, train_result = run_training(
    model=model,
    tokenizer=tokenizer,
    train_data=train_data,
    val_data=val_data,
    output_dir=OUTPUT_DIR,
)

[STAGE] Preparing train dataset...
[STAGE] Train dataset ready: 1228 examples
[STAGE] Preparing val dataset...
[STAGE] Val dataset ready: 154 examples
[STAGE] Creating trainer...


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1228 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/154 [00:00<?, ? examples/s]

[STAGE] Trainer created, calling trainer.train()...


...

[TRAIN] Step 1/77 | loss=0.5580 | lr=0.00e+00 | elapsed=6.2min | eta=471.9min
[TRAIN] Step 2/77 | loss=0.3409 | lr=5.00e-05 | elapsed=12.2min | eta=458.1min
[TRAIN] Step 3/77 | loss=0.1989 | lr=1.00e-04 | elapsed=18.1min | eta=447.6min
[TRAIN] Step 4/77 | loss=0.1805 | lr=1.50e-04 | elapsed=24.0min | eta=438.6min
[TRAIN] Step 5/77 | loss=0.3441 | lr=2.00e-04 | elapsed=30.6min | eta=441.3min
[TRAIN] Step 6/77 | loss=0.1283 | lr=2.00e-04 | elapsed=36.3min | eta=429.9min
[TRAIN] Step 7/77 | loss=0.2229 | lr=2.00e-04 | elapsed=42.9min | eta=428.7min
[TRAIN] Step 8/77 | loss=0.3244 | lr=1.99e-04 | elapsed=49.8min | eta=429.9min
[TRAIN] Step 9/77 | loss=0.4514 | lr=1.99e-04 | elapsed=56.6min | eta=427.3min
[TRAIN] Step 10/77 | loss=0.2062 | lr=1.98e-04 | elapsed=63.4min | eta=424.6min
[TRAIN] Step 11/77 | loss=0.1224 | lr=1.97e-04 | elapsed=69.5min | eta=417.0min
[TRAIN] Step 12/77 | loss=0.2936 | lr=1.95e-04 | elapsed=75.7min | eta=410.3min
[TRAIN] Step 13/77 | loss=0.2580 | lr=1.94e-04 | e

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


[STAGE] Push complete.


In [ ]:
# Log training metrics
metrics = train_result.metrics
print(f"Training loss:     {metrics['train_loss']:.4f}")
print(f"Training runtime:  {metrics['train_runtime']:.1f}s")
print(f"Samples/second:    {metrics['train_samples_per_second']:.2f}")

# Evaluate on validation set
eval_metrics = trainer.evaluate()
print(f"\nValidation loss:   {eval_metrics['eval_loss']:.4f}")

Training loss:     0.2074
Training runtime:  30652.8s
Samples/second:    0.04


KeyError: 'train_steps'

## Training Results

Adapter pushed to HuggingFace Hub: [`alapedriza/mistral-7b-function-calling-adapter`](https://huggingface.co/alapedriza/mistral-7b-function-calling-adapter)

| Metric | Value |
| --- | --- |
| Train loss | 0.1889 |
| Val loss | 0.1838 |
| Total steps | 77 |
| Training time | 511 min |
| Effective batch size | 16 (1 x grad_accum) |
| Examples | 1228 train / 154 val |

Val loss lower than train loss indicates zero overfitting and strong generalisation. The model learned the structured output format (tool selection + JSON arguments) without memorising individual examples.

**Next step**: Run inference on the 139-example test set (notebook 04) to measure actual accuracy on tool selection, parameter extraction, and JSON validity.